## 1. Install Dependencies

In [1]:
%pip install -q transformers==4.48.0 datasets peft trl bitsandbytes accelerate torch sentencepiece
print("✅ Dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 24.9 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 95.7 MB/s eta 0:00:00
✅ Dependencies installed


## 2. Imports & Configuration

In [2]:
import os
import json
import torch
from pathlib import Path
from dataclasses import dataclass

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

print(f"Torch version   : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Torch version   : 2.9.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB


In [3]:
@dataclass
class FinetuneConfig:
    # Model
    model_id: str = "Qwen/Qwen2.5-1.5B-Instruct"
    use_4bit: bool = True                    # 4-bit quantization (QLoRA)

    # Dataset
    dataset_path: str = "../notebook/query_optimizer_dataset.jsonl"
    max_seq_length: int = 256
    train_split: float = 0.9                 # 90% train / 10% eval

    # LoRA
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: tuple = (           # Qwen2 attention + MLP projections
        "q_proj", "k_proj", "v_proj",
        "o_proj", "gate_proj", "up_proj", "down_proj",
    )

    # Training
    output_dir: str = "./outputs/qwen-query-optimizer"
    num_epochs: int = 3
    per_device_train_batch_size: int = 4
    per_device_eval_batch_size: int = 4
    gradient_accumulation_steps: int = 4     # effective batch = 16
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.05
    lr_scheduler_type: str = "cosine"
    fp16: bool = not torch.cuda.is_bf16_supported()
    bf16: bool = torch.cuda.is_bf16_supported()
    logging_steps: int = 10
    eval_strategy: str = "epoch"
    save_strategy: str = "epoch"
    save_total_limit: int = 2
    load_best_model_at_end: bool = True

cfg = FinetuneConfig()
print(f"Model           : {cfg.model_id}")
print(f"Quantization    : {'4-bit QLoRA' if cfg.use_4bit else 'full precision'}")
print(f"Output dir      : {cfg.output_dir}")
print(f"LoRA rank       : {cfg.lora_r}  |  alpha: {cfg.lora_alpha}")

Model           : Qwen/Qwen2.5-1.5B-Instruct
Quantization    : 4-bit QLoRA
Output dir      : ./outputs/qwen-query-optimizer
LoRA rank       : 16  |  alpha: 32


## 3. Load & Prepare Dataset

Each example in the JSONL file is a 3-turn chat (`system` → `user` → `assistant`).  
We apply the **Qwen2.5 chat template** to convert each conversation into a single formatted string, then split into train / eval.

In [ ]:
# Colab: Mount Google Drive and copy dataset to /content
import sys, shutil
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# ← adjust this path to where you placed the file inside your Google Drive
DRIVE_FILE_PATH = "My Drive/query_optimizer_dataset.jsonl"

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    src = Path("/content/drive") / DRIVE_FILE_PATH
    dst = Path("/content/query_optimizer_dataset.jsonl")

    if not src.exists():
        raise FileNotFoundError(
            f"File not found in Drive at: {src}\n"
            f"Please upload 'query_optimizer_dataset.jsonl' to your Google Drive "
            f"and update DRIVE_FILE_PATH above."
        )

    shutil.copy(src, dst)
    print(f"✅ Dataset copied to {dst}")
else:
    print("Not running in Colab — skipping Drive mount.")

Mounted at /content/drive
✅ Dataset copied to /content/query_optimizer_dataset.jsonl


In [ ]:
def load_jsonl(path: str) -> list[dict]:
    """Load a JSONL file and return list of message dicts."""
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def _find_dataset(filename: str = "query_optimizer_dataset.jsonl") -> Path:
    """Locate the dataset file across Colab, local notebook, and CWD."""
    candidates = [
        Path("/content") / filename,
        Path.cwd() / filename,          
        Path.cwd() / "notebook" / filename,
        Path.cwd().parent / "notebook" / filename,   
    ]
    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(
        f"Could not locate '{filename}'.\n"
        "  • Colab : run the Google Drive mount cell above first.\n"
        "  • Local : make sure the .jsonl file is in the same folder as this notebook.\n"
        "Searched:\n" + "\n".join(f"  {p}" for p in candidates)
    )


dataset_path = _find_dataset()
print(f"Dataset path    : {dataset_path}")

raw_data = load_jsonl(str(dataset_path))

print(f"Total examples  : {len(raw_data)}")
print(f"\nSample record:")
print(json.dumps(raw_data[0], indent=2))

Dataset path    : /content/query_optimizer_dataset.jsonl
Total examples  : 534

Sample record:
{
  "messages": [
    {
      "role": "system",
      "content": "You are a query optimization agent. Rewrite user queries into clear, retrieval-focused enterprise document search queries. Do not add new information. Do not hallucinate."
    },
    {
      "role": "user",
      "content": "can someone explain job families?"
    },
    {
      "role": "assistant",
      "content": "Retrieve official enterprise documentation related to job families including procedures, policies, and operational guidelines."
    }
  ]
}


In [8]:
# Load tokenizer first so we can apply the Qwen2.5 chat template
tokenizer = AutoTokenizer.from_pretrained(
    cfg.model_id,
    trust_remote_code=True,
    padding_side="right",   # required for SFT loss masking
)
tokenizer.pad_token = tokenizer.eos_token

print(f"Vocab size      : {tokenizer.vocab_size:,}")
print(f"EOS token       : {tokenizer.eos_token!r}  (id={tokenizer.eos_token_id})")
print(f"Chat template   : {'found' if tokenizer.chat_template else 'NOT found – check model'}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size      : 151,643
EOS token       : '<|im_end|>'  (id=151645)
Chat template   : found


In [10]:
def format_example(record: dict) -> dict:
    """
    Apply Qwen2.5 chat template to a messages record.
    Returns a dict with a single 'text' key ready for SFTTrainer.
    """
    messages = record["messages"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


formatted = [format_example(r) for r in raw_data]

# Verify a formatted sample
print("=== Formatted sample ===")
print(formatted[0]["text"])
print()

# Build HuggingFace Dataset and train/eval split
hf_dataset = Dataset.from_list(formatted)
split = hf_dataset.train_test_split(test_size=1 - cfg.train_split, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train examples  : {len(train_dataset)}")
print(f"Eval examples   : {len(eval_dataset)}")

=== Formatted sample ===
<|im_start|>system
You are a query optimization agent. Rewrite user queries into clear, retrieval-focused enterprise document search queries. Do not add new information. Do not hallucinate.<|im_end|>
<|im_start|>user
can someone explain job families?<|im_end|>
<|im_start|>assistant
Retrieve official enterprise documentation related to job families including procedures, policies, and operational guidelines.<|im_end|>


Train examples  : 480
Eval examples   : 54


## 4. Load Base Model + Apply LoRA (QLoRA)

In [11]:
# ── 4a. Quantisation config (skipped on CPU) ──────────────────────────────────
bnb_config = None
if cfg.use_4bit and torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if cfg.bf16 else torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    print("QLoRA: 4-bit NF4 quantisation enabled")
else:
    print("Running in full precision (CPU or no bitsandbytes)")

# ── 4b. Load base model ────────────────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if cfg.bf16 else torch.float16,
)
model.config.use_cache = False          # required for gradient checkpointing
model.config.pretraining_tp = 1         # disable tensor parallelism in training

if bnb_config:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

print(f"\nLoaded: {cfg.model_id}")

# Count trainable params before LoRA
total_params = sum(p.numel() for p in model.parameters())
print(f"Total params    : {total_params / 1e6:.1f} M")

QLoRA: 4-bit NF4 quantisation enabled


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Loaded: Qwen/Qwen2.5-1.5B-Instruct
Total params    : 888.6 M


In [12]:
# ── 4c. Apply LoRA adapters ────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    target_modules=list(cfg.lora_target_modules),
    lora_dropout=cfg.lora_dropout,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

# Summary of trainable vs frozen parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,}  ({100 * trainable / total:.2f}% of {total / 1e6:.1f} M)")
model.print_trainable_parameters()

Trainable params : 18,464,768  (2.04% of 907.1 M)
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 5. Configure & Run Training

In [16]:
from trl import SFTConfig  # SFTConfig replaces TrainingArguments in TRL >= 0.12

os.makedirs(cfg.output_dir, exist_ok=True)

# SFTConfig merges TrainingArguments + SFT-specific args (max_seq_length, packing, etc.)
training_args = SFTConfig(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    warmup_ratio=cfg.warmup_ratio,
    lr_scheduler_type=cfg.lr_scheduler_type,
    fp16=cfg.fp16,
    bf16=cfg.bf16,
    logging_steps=cfg.logging_steps,
    eval_strategy=cfg.eval_strategy,
    save_strategy=cfg.save_strategy,
    save_total_limit=cfg.save_total_limit,
    load_best_model_at_end=cfg.load_best_model_at_end,
    report_to="none",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit" if bnb_config else "adamw_torch",
    dataloader_pin_memory=False,
    # SFT-specific args (moved here from SFTTrainer)
    max_seq_length=cfg.max_seq_length,
    dataset_text_field="text",
    packing=False,
)

print("SFTConfig configured ✅")

SFTConfig configured ✅


In [17]:
# SFTTrainer — trains only on the assistant response tokens (completion-only)
response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,          # SFTConfig (includes max_seq_length, packing, etc.)
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,  # renamed from 'tokenizer' in TRL >= 0.12
    data_collator=collator,
)

print("SFTTrainer ready ✅")

Converting train dataset to ChatML:   0%|          | 0/480 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/480 [00:00<?, ? examples/s]

Converting train dataset to ChatML:   0%|          | 0/480 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/480 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/480 [00:00<?, ? examples/s]

...

Converting eval dataset to ChatML:   0%|          | 0/54 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/54 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/54 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/54 [00:00<?, ? examples/s]

SFTTrainer ready ✅


In [18]:
# ── Train ─────────────────────────────────────────────────────────────────────
train_result = trainer.train()

metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

print("\n=== Training complete ===")
print(f"  Train loss      : {metrics.get('train_loss', 'N/A'):.4f}")
print(f"  Train runtime   : {metrics.get('train_runtime', 0):.0f}s")
print(f"  Steps/sec       : {metrics.get('train_steps_per_second', 0):.2f}")

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,0.001600,0.015599
2,0.004800,0.000177
3,0.000200,0.000161


***** train metrics *****
  total_flos               =   806711GF
  train_loss               =     0.8755
  train_runtime            = 0:10:05.13
  train_samples_per_second =       2.38
  train_steps_per_second   =      0.149

=== Training complete ===
  Train loss      : 0.8755
  Train runtime   : 605s
  Steps/sec       : 0.15


## 6. Evaluate on Eval Set

In [19]:
eval_metrics = trainer.evaluate()
trainer.log_metrics("eval", eval_metrics)
trainer.save_metrics("eval", eval_metrics)

print(f"Eval loss       : {eval_metrics.get('eval_loss', 'N/A'):.4f}")
print(f"Perplexity      : {2 ** eval_metrics.get('eval_loss', 0):.2f}")

***** eval metrics *****
  eval_loss               =     0.0002
  eval_runtime            = 0:00:06.78
  eval_samples_per_second =      7.958
  eval_steps_per_second   =      2.063
Eval loss       : 0.0002
Perplexity      : 1.00


## 7. Quick Inference Test

Run a few sample queries through the fine-tuned model to confirm it rewrites them correctly.

In [20]:
SYSTEM_PROMPT = (
    "You are a query optimization agent. Rewrite user queries into clear, "
    "retrieval-focused enterprise document search queries. "
    "Do not add new information. Do not hallucinate."
)

TEST_QUERIES = [
    "how do i request time off?",
    "what's the refund policy?",
    "explain the onboarding process for new hires",
    "where can I find the IT security guidelines?",
]


def optimize_query(user_query: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_query},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the generated tokens (skip the prompt)
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


print("=== Inference test ===\n")
for q in TEST_QUERIES:
    optimized = optimize_query(q)
    print(f"Input  : {q}")
    print(f"Output : {optimized}")
    print()

=== Inference test ===



/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


=== Inference test ===



/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Input  : how do i request time off?
Output : Retrieve official enterprise documentation related to requesting time off including procedures, policies, and operational guidelines.

Input  : what's the refund policy?
Output : Retrieve official enterprise documentation related to the refund policy including procedures, policies, and operational guidelines.

Input  : explain the onboarding process for new hires
Output : Retrieve official enterprise documentation related to onboarding including procedures, policies, and operational guidelines.

Input  : where can I find the IT security guidelines?
Output : Retrieve official enterprise documentation related to the IT security guidelines including procedures, policies, and operational guidelines.



## 8. Save the Fine-Tuned Model

Saves the LoRA adapters + merged model locally.  
Optionally push to the Hugging Face Hub by setting `push_to_hub=True`.

In [24]:
HF_TOKEN = "YOUR_HF_TOKEN_HERE"

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ Logged in to Hugging Face")

✅ Logged in to Hugging Face


In [25]:
import shutil
if IS_COLAB:
    PROJECT_OUTPUT_DIR = Path("/content/drive/My Drive/slm_first/outputs/qwen-query-optimizer")
else:
    PROJECT_OUTPUT_DIR = Path("./outputs/qwen-query-optimizer")

ADAPTER_SAVE_DIR = str(PROJECT_OUTPUT_DIR / "lora-adapters")
MERGED_SAVE_DIR  = str(PROJECT_OUTPUT_DIR / "merged-model")

PROJECT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving to: {PROJECT_OUTPUT_DIR}")

# Save LoRA adapters only (small, ~20 MB)
trainer.model.save_pretrained(ADAPTER_SAVE_DIR)
tokenizer.save_pretrained(ADAPTER_SAVE_DIR)
print(f"✅ LoRA adapters saved → {ADAPTER_SAVE_DIR}")

# Merge adapters into base weights and save full model 
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_id,
    device_map="cpu",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_SAVE_DIR)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained(MERGED_SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_SAVE_DIR)
print(f"✅ Merged model saved  → {MERGED_SAVE_DIR}")

# Push to Hugging Face Hub 
hub_repo_id = "abi-commits/qwen-query-optimizer"

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)

# Create the repo if it doesn't exist yet
api.create_repo(repo_id=hub_repo_id, repo_type="model", exist_ok=True)

# Upload the merged model folder
api.upload_folder(
    folder_path=MERGED_SAVE_DIR,
    repo_id=hub_repo_id,
    repo_type="model",
    commit_message="Add fine-tuned Qwen2.5-1.5B query optimizer (merged weights)",
)
print(f"✅ Pushed merged model → https://huggingface.co/{hub_repo_id}")

print("\n✅ All done!")

Saving to: /content/drive/My Drive/slm_first/outputs/qwen-query-optimizer
✅ LoRA adapters saved → /content/drive/My Drive/slm_first/outputs/qwen-query-optimizer/lora-adapters
✅ Merged model saved  → /content/drive/My Drive/slm_first/outputs/qwen-query-optimizer/merged-model


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...rged-model/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...d-model/model.safetensors:   1%|          | 25.1MB / 3.09GB            

✅ Pushed merged model → https://huggingface.co/abi-commits/qwen-query-optimizer

✅ All done!
